In [7]:
from pathlib import Path
import sys

import torch
import torch.nn.functional as F
from tqdm import tqdm

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from utils import PoreNetworkPermeabilityModel

device = "cuda" if torch.cuda.is_available() else "cpu"
NETWORK_DIR = ROOT / "outputs" / "networks"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "device:", device)


ROOT: f:\PycharmProjects\micro_ct device: cuda


In [8]:
EPOCHS = 100
LR = 1e-3
HIDDEN = 64
LAYERS = 3
VAL_FRACTION = 0.2
PATIENCE = 15
SEED = 42
CHECKPOINT_PATH = MODEL_DIR / "gnn_pnm_best.pth"


In [9]:
paths = sorted(NETWORK_DIR.glob("network_train_*.pt"))
if not paths:
    paths = sorted(NETWORK_DIR.glob("*.pt"))
if not paths:
    raise FileNotFoundError(f"No network .pt files found in {NETWORK_DIR}")
print("network files:", len(paths))

def load_network(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

networks = [load_network(path) for path in paths]
first = networks[0]
node_in = first.node_attr.shape[1]
edge_in = first.edge_attr.shape[1]

if len(networks) > 1:
    generator = torch.Generator().manual_seed(SEED)
    order = torch.randperm(len(networks), generator=generator).tolist()
    val_count = max(1, int(round(len(networks) * VAL_FRACTION)))
    val_count = min(val_count, len(networks) - 1)
else:
    order = [0]
    val_count = 0

val_indices = set(order[:val_count])
train_networks = [network for idx, network in enumerate(networks) if idx not in val_indices]
val_networks = [network for idx, network in enumerate(networks) if idx in val_indices]
train_paths = [path for idx, path in enumerate(paths) if idx not in val_indices]
val_paths = [path for idx, path in enumerate(paths) if idx in val_indices]

print("node_in:", node_in, "edge_in:", edge_in)
print("train networks:", len(train_networks), "val networks:", len(val_networks))


network files: 20
node_in: 21 edge_in: 16


In [10]:
model = PoreNetworkPermeabilityModel(node_in=node_in, edge_in=edge_in, hidden=HIDDEN, layers=LAYERS, mu=1.0e-3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
print("parameters:", sum(p.numel() for p in model.parameters()))


parameters: 102081


In [11]:
def target_k_from_metadata(network):
    k = network.metadata.get("openpnm_k")
    if k is None:
        raise ValueError("No target k found. Add LBM/lab/OpenPNM target to network.metadata['openpnm_k'].")
    return torch.tensor([k["kx"], k["ky"], k["kz"]], dtype=torch.float32)

def network_loss(network):
    network = network.to(device)
    target_k = target_k_from_metadata(network).to(device).clamp_min(1e-30)
    pred_k, log_g = model(
        network.node_attr,
        network.edge_index,
        network.edge_attr,
        network.coords,
        network.domain_size,
        log_g_hp=network.log_g_hp,
    )
    loss = F.smooth_l1_loss(torch.log(pred_k.clamp_min(1e-30)), torch.log(target_k))
    return loss, pred_k.detach(), target_k.detach(), log_g.detach()


In [12]:
def evaluate_networks(split_networks):
    if not split_networks:
        return None, None, None
    model.eval()
    total_loss = 0.0
    last_pred = None
    last_target = None
    with torch.no_grad():
        for network in split_networks:
            loss, pred_k, target_k, _ = network_loss(network)
            total_loss += float(loss.detach().cpu())
            last_pred = pred_k.detach().cpu()
            last_target = target_k.detach().cpu()
    return total_loss / len(split_networks), last_pred, last_target

best_loss = float("inf")
bad_epochs = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    last_train_pred = None
    last_train_target = None
    generator = torch.Generator().manual_seed(SEED + epoch)
    train_order = torch.randperm(len(train_networks), generator=generator).tolist()

    for idx in tqdm(train_order, desc=f"epoch {epoch}"):
        network = train_networks[idx]
        optimizer.zero_grad(set_to_none=True)
        loss, pred_k, target_k, _ = network_loss(network)
        loss.backward()
        optimizer.step()
        epoch_loss += float(loss.detach().cpu())
        last_train_pred = pred_k.detach().cpu()
        last_train_target = target_k.detach().cpu()
    train_loss = epoch_loss / max(len(train_networks), 1)
    val_loss, val_pred, val_target = evaluate_networks(val_networks)
    monitor_loss = val_loss if val_loss is not None else train_loss

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    if val_loss is None:
        print(f"epoch={epoch} train_loss={train_loss:.6f}")
        print("last train pred:", last_train_pred, "target:", last_train_target)
    else:
        print(f"epoch={epoch} train_loss={train_loss:.6f} val_loss={val_loss:.6f}")
        print("val example pred:", val_pred, "target:", val_target)

    if monitor_loss < best_loss:
        best_loss = monitor_loss
        bad_epochs = 0
        torch.save({
            "checkpoint_type": "gnn_pnm",
            "target_name": "openpnm_k",
            "model_state_dict": model.state_dict(),
            "node_in": node_in,
            "edge_in": edge_in,
            "hidden": HIDDEN,
            "layers": LAYERS,
            "epoch": epoch,
            "monitor_loss": monitor_loss,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "history": history,
            "train_files": [str(path.relative_to(ROOT)) for path in train_paths],
            "val_files": [str(path.relative_to(ROOT)) for path in val_paths],
        }, CHECKPOINT_PATH)
        print("saved:", CHECKPOINT_PATH)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stop at epoch={epoch}; best_loss={best_loss:.6f}")
            break


epoch 1: 100%|██████████| 20/20 [00:00<00:00, 21.15it/s]


epoch=1 loss=2.857367
last pred: tensor([2.0331e-12, 3.3884e-13, 3.1618e-12]) target: tensor([1.0016e-13, 8.5205e-14, 9.0502e-14])
saved: f:\PycharmProjects\micro_ct\models\gnn_pnm_best.pth


epoch 2: 100%|██████████| 20/20 [00:00<00:00, 21.55it/s]


epoch=2 loss=1.961545
last pred: tensor([3.2638e-13, 4.7362e-14, 5.7811e-13]) target: tensor([1.0016e-13, 8.5205e-14, 9.0502e-14])
saved: f:\PycharmProjects\micro_ct\models\gnn_pnm_best.pth


epoch 3: 100%|██████████| 20/20 [00:00<00:00, 22.28it/s]


...

epoch=14 loss=0.713666
last pred: tensor([3.9087e-13, 6.1299e-14, 6.6815e-13]) target: tensor([1.0016e-13, 8.5205e-14, 9.0502e-14])
saved: f:\PycharmProjects\micro_ct\models\gnn_pnm_best.pth


epoch 15: 100%|██████████| 20/20 [00:00<00:00, 21.79it/s]


epoch=15 loss=0.708145
last pred: tensor([3.6706e-13, 5.7592e-14, 6.2877e-13]) target: tensor([1.0016e-13, 8.5205e-14, 9.0502e-14])
saved: f:\PycharmProjects\micro_ct\models\gnn_pnm_best.pth


epoch 16:  20%|██        | 4/20 [00:00<00:00, 19.54it/s]


KeyboardInterrupt: 